# GameTheory-4-NashEquilibrium (C#)

**Navigation** : [<< 3-Topology2x2](GameTheory-3-Topology2x2.ipynb) | [Index](README.md) | [5-ZeroSum-Minimax >>](GameTheory-5-ZeroSum-Minimax-Csharp.ipynb)

**Twin C# (.NET Interactive)** de [GameTheory-4-NashEquilibrium.ipynb](GameTheory-4-NashEquilibrium.ipynb) — marathon **#4956** (parite .NET <-> Python), axe-2 SOTA **#3801** Prong B.

Le notebook Python invoque **nashpy** (+ `scipy.optimize`, `numpy`) pour calculer les equilibres de Nash via support enumeration, Lemke-Howson et vertex enumeration. Ce twin **deroule les algorithmes a la main** (BCL .NET 9, **0 NuGet, 0 dependance externe**) : on implemente l'enumeration des supports avec **elimination de Gauss from-scratch** pour resoudre les systemes d'indifference. L'interieur de la boite noire nashpy devient visible.

## Plan pedagogique

1. **Definition formelle** — equilibre de Nash (purs et mixtes)
2. **Equilibres purs** — enumeration par meilleure reponse mutuelle
3. **Strategies mixtes** — condition d'indifference, formule fermee 2x2
4. **Support enumeration (coeur algorithmique)** — elimination de Gauss, general m x n
5. **Jeux canoniques** — Prisonnier, Stag Hunt, Matching Pennies, Battle of the Sexes
6. **Pierre-Feuille-Ciseaux 3x3** — equilibre uniforme, zero-sum symetrique
7. **Exercices**

> **Prong B (#3801)** : ou la Python s'appuie sur nashpy (boite noire C), le C# rend l'algorithme explicite. La complémentarite pedagogique : l'eleve voit les etapes (choix des supports -> systeme lineaire -> elimination de Gauss -> validation dans le simplexe) que nashpy opaque.


## 1. Definition formelle

Un **equilibre de Nash** est un profil de strategies $\sigma^* = (\sigma_1^*, \sigma_2^*)$ tel qu'aucun joueur n'a interet a devier unilateralement :

$$u_i(\sigma_i^*, \sigma_{-i}^*) \geq u_i(\sigma_i, \sigma_{-i}^*) \quad \forall \sigma_i$$

- **Strategies pures** : chaque joueur choisit une action deterministe. Un equilibre pur est une paire $(i^*, j^*)$ telle que $i^*$ est meilleure reponse a $j^*$ et vice-versa.
- **Strategies mixtes** : chaque joueur randomise sur un sous-ensemble d'actions (le *support*). A l'equilibre, toute action du support doit etre **indifferentielle** (meme gain espere).

Le theoreme de Nash (1950) garantit l'existence d'au moins un equilibre dans tout jeu fini (par point fixe de Brouwer). Le defi algorithmique est de les **trouver**.


### 1.1 Classe `Game` — representation d'un jeu bimatriciel

Un jeu 2 joueurs en forme normale est defini par deux matrices de gains $A$ (Row) et $B$ (Col), de memes dimensions $m \times n$. `Payoffs` calcule le gain espere pour des strategies mixtes $\sigma_{row}$ (taille $m$) et $\sigma_{col}$ (taille $n$) : $u_{row} = \sigma_{row}^\top A \, \sigma_{col}$.

In [1]:
// Helpers d'affichage
using System.Linq;
using System.Text;

static string FI(double x, string fmt = "F4") => x.ToString(fmt);
static void Show(string s) { s.Display(); }

// Jeu bimatriciel (forme normale), m actions Row x n actions Col.
public class Game
{
    public double[,] A { get; }   // gains Row (m x n)
    public double[,] B { get; }   // gains Col (m x n)
    public string[] RowLabels { get; }
    public string[] ColLabels { get; }
    public string Name { get; }
    public int M => A.GetLength(0);
    public int N => A.GetLength(1);

    public Game(double[,] a, double[,] b, string[] rowLabels = null,
                string[] colLabels = null, string name = "Game")
    {
        A = a; B = b;
        RowLabels = rowLabels ?? DefaultLabels(M, "R");
        ColLabels = colLabels ?? DefaultLabels(N, "C");
        Name = name;
    }

    static string[] DefaultLabels(int n, string p) =>
        Enumerable.Range(0, n).Select(i => p + i).ToArray();

    // Gain espere : u_row = sigma_row^T . A . sigma_col
    public (double uRow, double uCol) Payoffs(double[] sigmaRow, double[] sigmaCol)
    {
        double ur = 0, uc = 0;
        for (int i = 0; i < M; i++)
            for (int j = 0; j < N; j++)
            {
                ur += sigmaRow[i] * A[i, j] * sigmaCol[j];
                uc += sigmaRow[i] * B[i, j] * sigmaCol[j];
            }
        return (ur, uc);
    }

    // Meilleure reponse de Row a sigma_col : vecteur indicateur de l'action maximisant A.sigma_col
    public double[] BestResponseRow(double[] sigmaCol)
    {
        double[] exp = new double[M];
        for (int i = 0; i < M; i++)
            for (int j = 0; j < N; j++)
                exp[i] += A[i, j] * sigmaCol[j];
        double max = exp.Max();
        double[] br = new double[M];
        for (int i = 0; i < M; i++) br[i] = Math.Abs(exp[i] - max) < 1e-9 ? 1.0 : 0.0;
        return br;
    }

    public void Display()
    {
        var sb = new StringBuilder();
        sb.AppendLine($"### {Name}  ({M}x{N})");
        sb.AppendLine();
        sb.AppendLine($"| Row \\ Col | {string.Join(" | ", ColLabels.Select(c => $"**{c}**"))} |");
        sb.AppendLine($"|---|{string.Join("|", Enumerable.Repeat("---", N))}|");
        for (int i = 0; i < M; i++)
        {
            var cells = new List<string> { $"**{RowLabels[i]}**" };
            for (int j = 0; j < N; j++)
                cells.Add($"({FI(A[i, j], "G4")}, {FI(B[i, j], "G4")})");
            sb.AppendLine(string.Join(" | ", cells) + " |");
        }
        sb.ToString().Display();  // affiche le tableau markdown
    }
}
Show("Classe Game chargee (matrices A, B, Payoffs, BestResponseRow, Display).");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Classe Game chargee (matrices A, B, Payoffs, BestResponseRow, Display).

## 2. Equilibres de Nash purs

Un equilibre pur $(i^*, j^*)$ verifie : $i^*$ est meilleure reponse de Row a $j^*$ **et** $j^*$ est meilleure reponse de Col a $i^*$. On enumerer toutes les paires et on teste cette condition.

In [2]:
// Equilibres de Nash purs par enumeration (mutual best response).
public static List<(int i, int j)> FindPureNash(Game g)
{
    var eqs = new List<(int, int)>();
    for (int i = 0; i < g.M; i++)
        for (int j = 0; j < g.N; j++)
        {
            // i est meilleure réponse de Row à j ?
            bool rowBr = true;
            for (int k = 0; k < g.M; k++)
                if (g.A[k, j] > g.A[i, j] + 1e-9) { rowBr = false; break; }
            // j est meilleure réponse de Col à i ?
            bool colBr = true;
            for (int l = 0; l < g.N; l++)
                if (g.B[i, l] > g.B[i, j] + 1e-9) { colBr = false; break; }
            if (rowBr && colBr) eqs.Add((i, j));
        }
    return eqs;
}

public static void ShowPureNash(Game g)
{
    g.Display();
    var pure = FindPureNash(g);
    var txt = $"Equilibres purs : {pure.Count}";
    foreach (var (i, j) in pure)
        txt += $"\n  ({g.RowLabels[i]}, {g.ColLabels[j]}) -> gains ({FI(g.A[i, j], "G4")}, {FI(g.B[i, j], "G4")})";
    if (pure.Count == 0) txt += "\n  (aucun -- il faudra un equilibre mixte)";
    txt.Display();
}

// Dilemme du Prisonnier : T=5 > R=3 > P=1 > S=0. (D,D) est l'unique equilibre.
var pd = new Game(new double[,] { { 3, 0 }, { 5, 1 } },
                  new double[,] { { 3, 5 }, { 0, 1 } },
                  new[] { "C", "D" }, new[] { "C", "D" }, "Prisoner's Dilemma");
ShowPureNash(pd);


### Prisoner's Dilemma  (2x2)

| Row \ Col | **C** | **D** |
|---|---|---|
**C** | (3, 3) | (0, 5) |
**D** | (5, 0) | (1, 1) |


Equilibres purs : 1
  (D, D) -> gains (1, 1)

In [3]:
// Stag Hunt : 2 equilibres purs (Stag,Stag) et (Hare,Hare) + 1 mixte.
var stag = new Game(new double[,] { { 4, 0 }, { 3, 3 } },
                    new double[,] { { 4, 3 }, { 0, 3 } },
                    new[] { "S", "H" }, new[] { "S", "H" }, "Stag Hunt");
ShowPureNash(stag);

// Coordination : 2 equilibres purs + 1 mixte.
var coord = new Game(new double[,] { { 2, 0 }, { 0, 1 } },
                     new double[,] { { 2, 0 }, { 0, 1 } },
                     new[] { "L", "R" }, new[] { "L", "R" }, "Coordination");
ShowPureNash(coord);


### Stag Hunt  (2x2)

| Row \ Col | **S** | **H** |
|---|---|---|
**S** | (4, 4) | (0, 3) |
**H** | (3, 0) | (3, 3) |


Equilibres purs : 2
  (S, S) -> gains (4, 4)
  (H, H) -> gains (3, 3)

### Coordination  (2x2)

| Row \ Col | **L** | **R** |
|---|---|---|
**L** | (2, 2) | (0, 0) |
**R** | (0, 0) | (1, 1) |


Equilibres purs : 2
  (L, L) -> gains (2, 2)
  (R, R) -> gains (1, 1)

## 3. Strategies mixtes et condition d'indifference

Une **strategie mixte** est une distribution de probabilite sur les actions. Pour un jeu 2x2, on parametre :
- $\sigma_{row} = (p, 1-p)$ avec $p \in [0,1]$
- $\sigma_{col} = (q, 1-q)$ avec $q \in [0,1]$

### Condition d'indifference

A l'equilibre mixte interieur, chaque joueur doit etre **indifferent** entre les actions de son support. Pour Row :

$$A_{00} q + A_{01}(1-q) = A_{10} q + A_{11}(1-q)$$

ce qui donne, en isolant $q$ :

$$q^* = \frac{A_{11} - A_{01}}{A_{00} - A_{01} - A_{10} + A_{11}}$$

et symetriquement pour $p$ (en utilisant $B$). C'est la **formule fermee 2x2**.

In [4]:
// Equilibre mixte 2x2 par la condition d'indifference (formule fermee).
// Retourne (sigma_row=(p,1-p), sigma_col=(q,1-q)) ou null si pas d'interieur.
public static (double[] sigmaRow, double[] sigmaCol)? ComputeMixedNash2x2(Game g)
{
    if (g.M != 2 || g.N != 2) return null;
    double a00 = g.A[0, 0], a01 = g.A[0, 1], a10 = g.A[1, 0], a11 = g.A[1, 1];
    double b00 = g.B[0, 0], b01 = g.B[0, 1], b10 = g.B[1, 0], b11 = g.B[1, 1];

    // Indifference de Row (resout q) :
    double denomQ = a00 - a01 - a10 + a11;
    if (Math.Abs(denomQ) < 1e-12) return null;
    double q = (a11 - a01) / denomQ;

    // Indifference de Col (resout p) :
    double denomP = b00 - b10 - b01 + b11;
    if (Math.Abs(denomP) < 1e-12) return null;
    double p = (b11 - b10) / denomP;

    if (p < -1e-9 || p > 1 + 1e-9 || q < -1e-9 || q > 1 + 1e-9) return null;
    p = Math.Clamp(p, 0.0, 1.0);
    q = Math.Clamp(q, 0.0, 1.0);
    return (new[] { p, 1 - p }, new[] { q, 1 - q });
}

// Matching Pennies : zero-sum, unique equilibre mixte (0.5, 0.5).
var mp = new Game(new double[,] { { 1, -1 }, { -1, 1 } },
                  new double[,] { { -1, 1 }, { 1, -1 } },
                  new[] { "Pile", "Face" }, new[] { "Pile", "Face" }, "Matching Pennies");
mp.Display();
var mpMix = ComputeMixedNash2x2(mp);
var (sr, sc) = mpMix.Value;
var (ur, uc) = mp.Payoffs(sr, sc);
$"- Row : p(Pile) = {FI(sr[0])}, p(Face) = {FI(sr[1])}".Display();
$"- Col : q(Pile) = {FI(sc[0])}, q(Face) = {FI(sc[1])}".Display();
$"- Gains : ({FI(ur)}, {FI(uc)})".Display();
"Verdict : Matching Pennies -> (0.5, 0.5) pour les deux joueurs (canonique).".Display();


### Matching Pennies  (2x2)

| Row \ Col | **Pile** | **Face** |
|---|---|---|
**Pile** | (1, -1) | (-1, 1) |
**Face** | (-1, 1) | (1, -1) |


- Row : p(Pile) = 0,5000, p(Face) = 0,5000

- Col : q(Pile) = 0,5000, q(Face) = 0,5000

- Gains : (0,0000, 0,0000)

Verdict : Matching Pennies -> (0.5, 0.5) pour les deux joueurs (canonique).

In [5]:
// Battle of the Sexes : 2 equilibres purs + 1 mixte.
var bos = new Game(new double[,] { { 2, 0 }, { 0, 1 } },
                   new double[,] { { 1, 0 }, { 0, 2 } },
                   new[] { "Opera", "Foot" }, new[] { "Opera", "Foot" }, "Battle of the Sexes");
bos.Display();
var bosPure = FindPureNash(bos);
$"Equilibres purs : {bosPure.Count} -> {string.Join(", ", bosPure.Select(e => $"({bos.RowLabels[e.i]},{bos.ColLabels[e.j]})"))}".Display();
var bosMix = ComputeMixedNash2x2(bos);
var (br, bc) = bosMix.Value;
$"Equilibre mixte : Row joue Opera avec p = {FI(br[0])} (2/3), Col joue Opera avec q = {FI(bc[0])} (1/3)".Display();

// Stag Hunt : verifier l'equilibre mixte a p = q = 0.75.
var stagMix = ComputeMixedNash2x2(stag);
var (ssr, ssc) = stagMix.Value;
$"Stag Hunt mixte : p(S) = {FI(ssr[0])} (0.75), q(S) = {FI(ssc[0])} (0.75)".Display();


### Battle of the Sexes  (2x2)

| Row \ Col | **Opera** | **Foot** |
|---|---|---|
**Opera** | (2, 1) | (0, 0) |
**Foot** | (0, 0) | (1, 2) |


Equilibres purs : 2 -> (Opera,Opera), (Foot,Foot)

Equilibre mixte : Row joue Opera avec p = 0,6667 (2/3), Col joue Opera avec q = 0,3333 (1/3)

Stag Hunt mixte : p(S) = 0,7500 (0.75), q(S) = 0,7500 (0.75)

## 4. Support enumeration (coeur algorithmique, from-scratch)

La formule fermee ne marche que pour 2x2. Pour le cas general, nashpy utilise la **support enumeration** : pour chaque paire de supports $(S_{row}, S_{col})$ de meme taille $k$, on cherche des strategies mixtes qui rendent chaque joueur indifferent entre les actions de son support, puis on verifie la validite (probabilites $\geq 0$, actions hors-support non profitables).

### Pourquoi l'elimination de Gauss ?

Pour un support de taille $k$, l'indifference impose $k-1$ equations lineaires (actions du support de gain egal) + 1 equation de normalisation ($\sum p_i = 1$). On resout un systeme lineaire $k \times k$ : c'est la qu'intervient **l'elimination de Gauss** (que nashpy resout via numpy/scipy en interne).

On implemente `SolveLinearSystem` (Gauss avec pivot partiel) puis `SupportEnumeration`.

In [6]:
// Resolution d'un systeme lineaire Ax = b par elimination de Gauss avec pivot partiel.
// Retourne null si le systeme est singulier.
public static double[] SolveLinearSystem(double[,] A, double[] b)
{
    int n = b.Length;
    // Copie augmentee [A | b]
    double[,] m = new double[n, n + 1];
    for (int i = 0; i < n; i++)
    {
        for (int j = 0; j < n; j++) m[i, j] = A[i, j];
        m[i, n] = b[i];
    }
    // Elimination
    for (int col = 0; col < n; col++)
    {
        // Pivot partiel : plus grande valeur absolue
        int piv = col;
        for (int r = col + 1; r < n; r++)
            if (Math.Abs(m[r, col]) > Math.Abs(m[piv, col])) piv = r;
        if (Math.Abs(m[piv, col]) < 1e-12) return null;  // singulier
        if (piv != col)
            for (int c = 0; c <= n; c++) (m[col, c], m[piv, c]) = (m[piv, c], m[col, c]);
        // Elimination sous le pivot
        for (int r = col + 1; r < n; r++)
        {
            double f = m[r, col] / m[col, col];
            for (int c = col; c <= n; c++) m[r, c] -= f * m[col, c];
        }
    }
    // Substitution remontee
    double[] x = new double[n];
    for (int i = n - 1; i >= 0; i--)
    {
        double s = m[i, n];
        for (int j = i + 1; j < n; j++) s -= m[i, j] * x[j];
        x[i] = s / m[i, i];
    }
    return x;
}

// Test rapide : resoud le systeme [[2,1],[1,3]] x = [3,5] -> x = (0.8, 1.4)
var test = SolveLinearSystem(new double[,] { { 2, 1 }, { 1, 3 } }, new[] { 3.0, 5.0 });
$"Test Gauss : x = ({FI(test[0], "G4")}, {FI(test[1], "G4")})  [attendu (0.8, 1.4)]".Display();


Test Gauss : x = (0,8, 1,4)  [attendu (0.8, 1.4)]

In [7]:
// Generateur de tous les sous-ensembles de taille k de {0..n-1}.
static IEnumerable<int[]> Combinations(int n, int k)
{
    int[] c = Enumerable.Range(0, k).ToArray();
    while (true)
    {
        yield return (int[])c.Clone();
        int i = k - 1;
        while (i >= 0 && c[i] == n - k + i) i--;
        if (i < 0) yield break;
        c[i]++;
        for (int j = i + 1; j < k; j++) c[j] = c[j - 1] + 1;
    }
}

// Support enumeration complet (general m x n).
// Pour chaque (S_row, S_col) de meme taille k, on resout le systeme d'indifference
// par elimination de Gauss, puis on valide (probas >= 0, pas de deviation profitable).
public record NashEq(double[] SigmaRow, double[] SigmaCol);

public static List<NashEq> SupportEnumeration(Game g)
{
    var result = new List<NashEq>();
    int maxK = Math.Min(g.M, g.N);
    for (int k = 1; k <= maxK; k++)
    {
        foreach (var sR in Combinations(g.M, k))
        foreach (var sC in Combinations(g.N, k))
        {
            var eq = SolveSupport(g, sR, sC, k);
            if (eq != null && !ContainsApprox(result, eq)) result.Add(eq);
        }
    }
    return result;
}

// Resout le systeme d'indifference pour un couple de supports donne.
static NashEq SolveSupport(Game g, int[] sR, int[] sC, int k)
{
    // --- sigma_col (sur sC) : indifference des k actions de sR face a sigma_col ---
    // Pour chaque paire consecutive (sR[t], sR[t+1]) : gain(sR[t]) = gain(sR[t+1])
    // k-1 equations + 1 normalisation sum=1 -> systeme k x k en les inconnues sigma_col[sC[0..k-1]]
    double[,] Ac = new double[k, k];
    double[] bc = new double[k];
    for (int t = 0; t < k - 1; t++)
    {
        int a1 = sR[t], a2 = sR[t + 1];
        for (int j = 0; j < k; j++)
        {
            Ac[t, j] = g.A[a1, sC[j]] - g.A[a2, sC[j]];
        }
        bc[t] = 0;
    }
    for (int j = 0; j < k; j++) Ac[k - 1, j] = 1.0;
    bc[k - 1] = 1.0;
    var sigmaCdense = SolveLinearSystem(Ac, bc);
    if (sigmaCdense == null) return null;

    // --- sigma_row (sur sR) : indifference des k actions de sC ---
    double[,] Ar = new double[k, k];
    double[] br = new double[k];
    for (int t = 0; t < k - 1; t++)
    {
        int a1 = sC[t], a2 = sC[t + 1];
        for (int i = 0; i < k; i++)
            Ar[t, i] = g.B[sR[i], a1] - g.B[sR[i], a2];
        br[t] = 0;
    }
    for (int i = 0; i < k; i++) Ar[k - 1, i] = 1.0;
    br[k - 1] = 1.0;
    var sigmaRdense = SolveLinearSystem(Ar, br);
    if (sigmaRdense == null) return null;

    // Validite : toutes les probas >= 0
    if (sigmaCdense.Any(v => v < -1e-7)) return null;
    if (sigmaRdense.Any(v => v < -1e-7)) return null;

    // Reconstituer les vecteurs complets
    double[] sc = new double[g.N], sr = new double[g.M];
    for (int j = 0; j < k; j++) sc[sC[j]] = Math.Max(0, sigmaCdense[j]);
    for (int i = 0; i < k; i++) sr[sR[i]] = Math.Max(0, sigmaRdense[i]);

    // Aucune deviation profitable : actions hors-support ne doivent pas battre le gain indifferent
    double gR = 0; for (int j = 0; j < g.N; j++) gR += g.A[sR[0], j] * sc[j];
    for (int i = 0; i < g.M; i++)
    {
        if (sR.Contains(i)) continue;
        double dev = 0; for (int j = 0; j < g.N; j++) dev += g.A[i, j] * sc[j];
        if (dev > gR + 1e-7) return null;
    }
    double gC = 0; for (int i = 0; i < g.M; i++) gC += g.B[i, sC[0]] * sr[i];
    for (int j = 0; j < g.N; j++)
    {
        if (sC.Contains(j)) continue;
        double dev = 0; for (int i = 0; i < g.M; i++) dev += g.B[i, j] * sr[i];
        if (dev > gC + 1e-7) return null;
    }
    return new NashEq(sr, sc);
}

static bool ContainsApprox(List<NashEq> list, NashEq eq)
{
    foreach (var e in list)
    {
        bool same = e.SigmaRow.Zip(eq.SigmaRow).All(p => Math.Abs(p.First - p.Second) < 1e-6)
                 && e.SigmaCol.Zip(eq.SigmaCol).All(p => Math.Abs(p.First - p.Second) < 1e-6);
        if (same) return true;
    }
    return false;
}
"Support enumeration from-scratch (Gauss + Combinations + SolveSupport) charge.".Display();


Support enumeration from-scratch (Gauss + Combinations + SolveSupport) charge.

## 5. Analyse complete sur les jeux canoniques

On applique `SupportEnumeration` a tous les jeux. Le verdict doit matcher nashpy : Prisonnier -> 1 equilibre pur (D,D), Stag Hunt -> 2 purs + 1 mixte, Matching Pennies -> 1 mixte (0.5/0.5), Battle of the Sexes -> 2 purs + 1 mixte (2/3, 1/3).

In [8]:
public static void Analyze(Game g)
{
    g.Display();
    var eqs = SupportEnumeration(g);
    $"Nombre d'equilibres de Nash (support enum) : {eqs.Count}".Display();
    int idx = 1;
    foreach (var eq in eqs)
    {
        var (ur, uc) = g.Payoffs(eq.SigmaRow, eq.SigmaCol);
        string fmtSr = string.Join(", ", g.RowLabels.Select((lab, i) => $"{lab}={FI(eq.SigmaRow[i])}"));
        string fmtSc = string.Join(", ", g.ColLabels.Select((lab, j) => $"{lab}={FI(eq.SigmaCol[j])}"));
        $"  EQ{idx}: Row[{fmtSr}] | Col[{fmtSc}] -> gains ({FI(ur, "G4")}, {FI(uc, "G4")})".Display();
        idx++;
    }
}

Analyze(pd);
"---".Display();
Analyze(stag);


### Prisoner's Dilemma  (2x2)

| Row \ Col | **C** | **D** |
|---|---|---|
**C** | (3, 3) | (0, 5) |
**D** | (5, 0) | (1, 1) |


Nombre d'equilibres de Nash (support enum) : 1

  EQ1: Row[C=0,0000, D=1,0000] | Col[C=0,0000, D=1,0000] -> gains (1, 1)

---

### Stag Hunt  (2x2)

| Row \ Col | **S** | **H** |
|---|---|---|
**S** | (4, 4) | (0, 3) |
**H** | (3, 0) | (3, 3) |


Nombre d'equilibres de Nash (support enum) : 3

  EQ1: Row[S=1,0000, H=0,0000] | Col[S=1,0000, H=0,0000] -> gains (4, 4)

  EQ2: Row[S=0,0000, H=1,0000] | Col[S=0,0000, H=1,0000] -> gains (3, 3)

  EQ3: Row[S=0,7500, H=0,2500] | Col[S=0,7500, H=0,2500] -> gains (3, 3)

In [9]:
Analyze(coord);
"---".Display();
Analyze(mp);
"---".Display();
Analyze(bos);


### Coordination  (2x2)

| Row \ Col | **L** | **R** |
|---|---|---|
**L** | (2, 2) | (0, 0) |
**R** | (0, 0) | (1, 1) |


Nombre d'equilibres de Nash (support enum) : 3

  EQ1: Row[L=1,0000, R=0,0000] | Col[L=1,0000, R=0,0000] -> gains (2, 2)

  EQ2: Row[L=0,0000, R=1,0000] | Col[L=0,0000, R=1,0000] -> gains (1, 1)

  EQ3: Row[L=0,3333, R=0,6667] | Col[L=0,3333, R=0,6667] -> gains (0,6667, 0,6667)

...

### Battle of the Sexes  (2x2)

| Row \ Col | **Opera** | **Foot** |
|---|---|---|
**Opera** | (2, 1) | (0, 0) |
**Foot** | (0, 0) | (1, 2) |


Nombre d'equilibres de Nash (support enum) : 3

  EQ1: Row[Opera=1,0000, Foot=0,0000] | Col[Opera=1,0000, Foot=0,0000] -> gains (2, 1)

  EQ2: Row[Opera=0,0000, Foot=1,0000] | Col[Opera=0,0000, Foot=1,0000] -> gains (1, 2)

  EQ3: Row[Opera=0,6667, Foot=0,3333] | Col[Opera=0,3333, Foot=0,6667] -> gains (0,6667, 0,6667)

## 6. Pierre-Feuille-Ciseaux (3x3 zero-sum symetrique)

Le jeu RPS est zero-sum ($B = -A$) et symetrique. Le theoreme garanti que l'equilibre est **uniforme** : chaque joueur joue $(1/3, 1/3, 1/3)$, pour un gain nul. C'est un test non-trivial de notre support enumeration 3x3 (le systeme lineaire est de taille 3, resolu par Gauss).

In [10]:
var rps = new Game(
    new double[,] { { 0, -1, 1 }, { 1, 0, -1 }, { -1, 1, 0 } },
    new double[,] { { 0, 1, -1 }, { -1, 0, 1 }, { 1, -1, 0 } },
    new[] { "P", "F", "C" }, new[] { "P", "F", "C" }, "Rock-Paper-Scissors");
Analyze(rps);
"Verdict : RPS -> (1/3, 1/3, 1/3) pour les deux, gain 0. Confirme le theoreme.".Display();


### Rock-Paper-Scissors  (3x3)

| Row \ Col | **P** | **F** | **C** |
|---|---|---|---|
**P** | (0, 0) | (-1, 1) | (1, -1) |
**F** | (1, -1) | (0, 0) | (-1, 1) |
**C** | (-1, 1) | (1, -1) | (0, 0) |


Nombre d'equilibres de Nash (support enum) : 1

  EQ1: Row[P=0,3333, F=0,3333, C=0,3333] | Col[P=0,3333, F=0,3333, C=0,3333] -> gains (0, 0)

Verdict : RPS -> (1/3, 1/3, 1/3) pour les deux, gain 0. Confirme le theoreme.

### 6.1 Jeu asymetrique 3x2

Un jeu non-carre ou Row a 3 actions et Col 2. La support enumeration trouve tous les equilibres (purs et mixtes) sans modification de l'algorithme — c'est l'interet de la formulation generale.

In [11]:
var asym = new Game(
    new double[,] { { 3, 1 }, { 0, 4 }, { 2, 2 } },
    new double[,] { { 2, 3 }, { 4, 1 }, { 1, 2 } },
    new[] { "R0", "R1", "R2" }, new[] { "C0", "C1" }, "Asymetrique 3x2");
Analyze(asym);


### Asymetrique 3x2  (3x2)

| Row \ Col | **C0** | **C1** |
|---|---|---|
**R0** | (3, 2) | (1, 3) |
**R1** | (0, 4) | (4, 1) |
**R2** | (2, 1) | (2, 2) |


Nombre d'equilibres de Nash (support enum) : 2

  EQ1: Row[R0=0,7500, R1=0,2500, R2=0,0000] | Col[C0=0,5000, C1=0,5000] -> gains (2, 2,5)

  EQ2: Row[R0=0,0000, R1=0,2500, R2=0,7500] | Col[C0=0,5000, C1=0,5000] -> gains (2, 1,75)

## 7. Exercices

> **Convention C.1** : les stubs s'executent sans erreur (jamais `throw NotImplementedException`). Remplir le corps, re-executer, verifier.

### Exercice 1 — Elimination iterative des strategies strictement dominees

**Objectif** : implementer l'elimination iterative. Une action Row $i$ est strictement dominee s'il existe $i'$ tel que $A[i', j] > A[i, j]$ pour tout $j$. On elimine, puis on reitere sur la sous-matrice jusqu'a stabilite.

**Indices** :
- Etape 1 : trouver les lignes/colonnes dominees de la matrice courante.
- Etape 2 : les retirer, reiterer.
- Etape 3 : les profils survivants sont candidats a etre equilibres de Nash (pour le PD, on doit converger vers (D,D)).

In [12]:
// Exercice 1 : elimination iterative des strategies strictement dominees.
// TODO etudiant : implementer. Retourner la liste des indices (lignes, colonnes) survivants.
public static (List<int> survivingRow, List<int> survivingCol) IteratedDominance(double[,] A, double[,] B)
{
    // Indice : une action i est strictement dominee s'il existe i' tel que
    //   A[i', j] > A[i, j] pour TOUT j (sur les colonnes survivantes).
    // Eliminer puis reiterer jusqu'a stabilite.
    var sR = new List<int>();   // TODO etudiant
    var sC = new List<int>();   // TODO etudiant
    return (sR, sC);
}

// Test : Dilemme du Prisonnier -> doit converger vers (D, D) seul survivant.
var (rPD, cPD) = IteratedDominance(
    new double[,] { { 3, 0 }, { 5, 1 } },
    new double[,] { { 3, 5 }, { 0, 1 } });
$"PD survivants Row = [{string.Join(",", rPD)}], Col = [{string.Join(",", cPD)}]".Display();
"Exercice a completer".Display();


PD survivants Row = [], Col = []

Exercice a completer

### Exercice 2 — Pierre-Feuille-Ciseaux biaise

**Objectif** : construire un RPS ou la Pierre rapporte 2 (au lieu de 1) en cas de victoire, puis calculer l'equilibre mixte. Doit-il etre encore uniforme ?

**Indice** : modifier la matrice $A$ pour que `Pierre>Feuille` et `Pierre>Ciseaux` valent 2, puis appeler `SupportEnumeration`. Le verdict : l'equilibre n'est PAS uniforme (Pierre etant plus forte, les adversaires la jouent moins).

In [13]:
// Exercice 2 : RPS biaise (Pierre = gain 2).
// TODO etudiant : construire A_biased, lancer SupportEnumeration, mesurer l'ecart a uniforme.
public record BiasedRpsResult(double[,] A, double[] sigmaRow, bool isUniform);

static BiasedRpsResult ExoRpsBiaise()
{
    // TODO etudiant : construire la matrice biaisee et retourner le resultat.
    // Indice : A[0,1] et A[0,2] (Pierre bat Feuille / Ciseaux) = 2 au lieu de 1.
    return null;  // TODO etudiant
}

"Exercice a completer".Display();


Exercice a completer

### Exercice 3 — Stag Hunt parametrique

**Objectif** : faire varier le gain de cooporation $R \in \{1..9\}$ dans Stag Hunt et tracer l'evolution de $p^*$ (probabilite de jouer Stag a l'equilibre mixte). On doit observer que $p^* \to 1$ quand $R$ augmente (cooparer devient de plus en plus attractif).

In [14]:
// Exercice 3 : Stag Hunt parametrique.
// TODO etudiant : pour R in 1..9, calculer p* (formule fermee ou support enum)
// et collecter les valeurs. Verifier que p* croit avec R.
public static List<(int R, double pStar)> ParametricStagHunt()
{
    var results = new List<(int, double)>();
    // TODO etudiant : boucler sur R, construire la matrice Stag Hunt (gain R en (S,S), 3 sinon),
    // appeler ComputeMixedNash2x2, stocker p*.
    return results;  // TODO etudiant
}

"Exercice a completer".Display();


Exercice a completer

## Conclusion

### Ce que vous avez appris

- **Equilibre de Nash pur** : enumeration par meilleure reponse mutuelle (algorithme $O(mn \cdot (m+n))$).
- **Equilibre mixte 2x2** : formule fermee par condition d'indifference $q^* = (a_{11} - a_{01})/(a_{00} - a_{01} - a_{10} + a_{11})$.
- **Support enumeration general** : pour chaque couple de supports de meme taille, on resout un systeme lineaire (elimination de Gauss) puis on valide dans le simplexe. C'est l'algorithme exact que nashpy execute en interne.
- **Cas canoniques verifies** : Matching Pennies (0.5/0.5), Battle of the Sexes (2/3, 1/3), Stag Hunt (0.75, 0.75), Prisonnier (pur (D,D)), RPS 3x3 (uniforme 1/3).

### Pont avec la version Python

La version Python ([GameTheory-4-NashEquilibrium.ipynb](GameTheory-4-NashEquilibrium.ipynb)) invoque **nashpy** qui fournit `support_enumeration()`, `lemke_howson_enumeration()` et `vertex_enumeration()` comme boites noires. Ce twin C# deroule le `support_enumeration` a la main : elimination de Gauss + validation. Le notebook suivant ([GameTheory-5-ZeroSum-Minimax](GameTheory-5-ZeroSum-Minimax-Csharp.ipynb)) traite le cas zero-sum ou le simplexe (algorithme de von Neumann) remplace la support enumeration.

### Prong B (#3801)

Le twin illustre la complémentarite pedagogique : ou nashpy opaque l'algorithme, le C# l'expose. L'eleve voit **pourquoi** l'elimination de Gauss resout l'indifference et **comment** on valide qu'une solution du systeme lineaire est bien un equilibre de Nash (pas de deviation profitable hors-support).

---
*Marathon #4956 (parite .NET <-> Python) — axe SOTA #3801 Prong B.*
